# Vesti-Care -- Preprocess 3 datasets directly on Colab (from Drive)

Chay notebook nay TRUOC `train_models_colab_v2.ipynb`, khi ban da co san 3 bo
raw dataset (SisFall, KFall, VestibularDataset) tren Google Drive va muon
tranh phai tai ve may de chay lai `build_subject_independent_dataset.py`.

Notebook nay KHONG can GPU (chi CPU/numpy/scipy/pandas), nen chon runtime
thuong (khong can T4) de tiet kiem quota GPU cho notebook train.

No lam dung 1 viec: chep 5 file script hien co cua repo
(`preprocess.py`, `preprocess_kfall.py`, `preprocess_vestibular.py`,
`resampling.py`, `build_subject_independent_dataset.py`) vao `/content/repo`,
tro raw-data cua chung sang thu muc Drive cua ban qua symlink, chay
`build()`, roi chep ket qua `processed_3class_v2/` nguoc lai len Drive dung
vi tri ma `train_models_colab_v2.ipynb` da mong doi
(`My Drive/VestiCare/data_v2/`).

Khong co dong logic tien xu ly nao bi thay doi o day -- day chi la ban sao
nguyen van cua 5 file trong repo, chay tren Colab thay vi chay local.

In [ ]:
# Cell 1 -- Mount Drive (khong can GPU cho buoc nay)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2 -- CONFIG: tro dung 3 thu muc raw dataset cua ban tren Drive
#
# SUA 3 duong dan ben duoi cho khop voi noi ban da upload len Drive.
# Neu chua chac chan duong dan dung, chay cell nay -- no se bao loi ro rang
# neu thu muc khong ton tai, va liet ke vai thu muc con dau tien de doi chieu.

import os
from pathlib import Path

DRIVE_SISFALL_DIR    = '/content/drive/MyDrive/VestiCare/raw/SisFall_dataset'
DRIVE_KFALL_DIR      = '/content/drive/MyDrive/VestiCare/raw/KFALL_dataset'
DRIVE_VESTIBULAR_DIR = '/content/drive/MyDrive/VestiCare/raw/vestibular_dataset'

# Noi luu ket qua processed_3class_v2/ -- PHAI trung voi DRIVE_DATA_DIR trong
# train_models_colab_v2.ipynb (Cell 2) de notebook train doc duoc luon.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/VestiCare/data_v2'

_checks = {
    'SisFall (DATASET_DIR)':    DRIVE_SISFALL_DIR,
    'KFall (KFALL_DIR)':        DRIVE_KFALL_DIR,
    'VestibularDataset (VEST_DIR)': DRIVE_VESTIBULAR_DIR,
}

all_ok = True
for label, p in _checks.items():
    path = Path(p)
    if not path.is_dir():
        all_ok = False
        print(f'[MISSING] {label}: {p}')
        parent = path.parent
        if parent.is_dir():
            sample = sorted(x.name for x in parent.iterdir())[:10]
            print(f'          -- thu muc cha {parent} co: {sample}')
        else:
            print(f'          -- thu muc cha {parent} cung khong ton tai. '
                  f'Dung Files panel (bieu tuong thu muc ben trai) de tim '
                  f'duong dan chinh xac roi sua lai o tren.')
    else:
        sample = sorted(x.name for x in path.iterdir())[:5]
        print(f'[OK] {label}: {p}')
        print(f'     mau 5 muc con dau tien: {sample}')

if not all_ok:
    raise SystemExit('Sua lai 3 duong dan DRIVE_*_DIR o tren roi chay lai Cell 2.')


In [ ]:
# Cell 3 -- Tao khung thu muc /content/repo va cac __init__.py rong
#
# Luu y: day la nhung stub __init__.py rong de resampling.py import duoc doc lap.

REPO = Path('/content/repo')
(REPO / 'ml' / 'training').mkdir(parents=True, exist_ok=True)
(REPO / 'ml' / 'common').mkdir(parents=True, exist_ok=True)
(REPO / 'ml' / 'data' / 'raw').mkdir(parents=True, exist_ok=True)

for init_path in [
    REPO / 'ml' / 'training' / '__init__.py',
    REPO / 'ml' / 'common' / '__init__.py',
]:
    init_path.write_text('', encoding='utf-8')

print('Da tao khung thu muc + stub __init__.py tai', REPO)


### Cell 4-8: chep 5 file script (nguyen van tu repo) vao Colab

Moi cell ben duoi ghi de nguyen mot file. Khong sua tay -- neu ban da sua
logic tien xu ly trong repo, chay lai script builder o may local
(`build_preprocess_notebook.py`) de dong bo lai truoc khi day.

In [ ]:
# Cell 4 -- resampling.py (anti-aliased polyphase resampler)
%%writefile /content/repo/ml/common/resampling.py
"""Exact anti-aliased rational timebase conversion for legacy public IMU.

The legacy vestibular preprocessor decimated 128 Hz data with an integer stride
of ``128 // 50 == 2``, which produces 64 Hz and makes a nominal 100-sample
window span 1.5625 s instead of 2 s. This module replaces only that defect with
deterministic anti-aliased rational polyphase resampling: reduce the rate ratio
by its GCD, then apply ``scipy.signal.resample_poly`` with a fixed Kaiser window
and line padding. 128 Hz to 50 Hz uses exactly ``up=25`` and ``down=64``.

Output sample count is ``ceil(n * up / down)``. Timestamps sit on an exact
integer-nanosecond grid at the target rate and never reach the end of the source
span. Input dtype (float32 or float64) is preserved; non-finite, boolean,
object, and one-dimensional inputs are rejected.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Tuple

import numpy as np
from scipy.signal import resample_poly

RESAMPLING_SCHEMA_VERSION = 1

_KAISER_WINDOW = ("kaiser", 5.0)


class ResamplingError(ValueError):
    """Raised with one stable machine-readable code."""

    def __init__(self, code: str) -> None:
        super().__init__(code)
        self.code = code

    def __str__(self) -> str:  # pragma: no cover - trivial
        return self.code


@dataclass(frozen=True)
class RationalRate:
    source_hz: int
    target_hz: int
    up: int
    down: int


@dataclass(frozen=True)
class ResampledSeries:
    values: np.ndarray
    timestamp_ns: Tuple[int, ...]
    rate: RationalRate
    source_sample_count: int


def _check_positive_int(value: object, code: str) -> int:
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)):
        raise ResamplingError(code)
    number = int(value)
    if number <= 0:
        raise ResamplingError(code)
    return number


def _validate_input(values: np.ndarray) -> np.ndarray:
    if not isinstance(values, np.ndarray):
        raise ResamplingError("resample_input_not_ndarray")
    if values.dtype == np.bool_ or values.dtype.kind not in ("f",):
        raise ResamplingError("resample_input_dtype_invalid")
    if values.ndim != 2:
        raise ResamplingError("resample_input_rank_invalid")
    if values.shape[0] <= 0 or values.shape[1] <= 0:
        raise ResamplingError("resample_input_empty")
    if not np.all(np.isfinite(values)):
        raise ResamplingError("resample_input_not_finite")
    return values


def _rational_rate(source_hz: int, target_hz: int) -> RationalRate:
    divisor = math.gcd(source_hz, target_hz)
    return RationalRate(
        source_hz=source_hz,
        target_hz=target_hz,
        up=target_hz // divisor,
        down=source_hz // divisor,
    )


def _target_period_ns(target_hz: int) -> int:
    if 1_000_000_000 % target_hz != 0:
        raise ResamplingError("resample_target_rate_not_integer_ns")
    return 1_000_000_000 // target_hz


def resample_polyphase(
    values: np.ndarray,
    *,
    source_hz: int,
    target_hz: int,
    start_timestamp_ns: int,
) -> ResampledSeries:
    """Apply deterministic anti-aliased rational polyphase resampling."""

    source_hz = _check_positive_int(source_hz, "resample_source_rate_invalid")
    target_hz = _check_positive_int(target_hz, "resample_target_rate_invalid")
    if isinstance(start_timestamp_ns, bool) or not isinstance(start_timestamp_ns, int):
        raise ResamplingError("resample_start_timestamp_invalid")
    if start_timestamp_ns < 0:
        raise ResamplingError("resample_start_timestamp_invalid")

    array = _validate_input(values)
    in_dtype = array.dtype
    n = int(array.shape[0])

    rate = _rational_rate(source_hz, target_hz)
    period_ns = _target_period_ns(target_hz)

    expected_count = math.ceil(n * rate.up / rate.down)
    work = np.ascontiguousarray(array, dtype=np.float64)
    resampled = resample_poly(
        work, up=rate.up, down=rate.down, axis=0, window=_KAISER_WINDOW, padtype="line"
    )
    if resampled.shape[0] != expected_count:  # pragma: no cover - guarded by scipy contract
        raise ResamplingError("resample_output_count_unexpected")

    resampled = np.ascontiguousarray(resampled, dtype=in_dtype)
    if not np.all(np.isfinite(resampled)):
        raise ResamplingError("resample_output_not_finite")

    timestamps = tuple(
        int(start_timestamp_ns + index * period_ns) for index in range(expected_count)
    )
    source_span_ns = start_timestamp_ns + math.ceil(n * 1_000_000_000 / source_hz)
    if timestamps and timestamps[-1] >= source_span_ns:
        raise ResamplingError("resample_timestamp_exceeds_source_span")

    return ResampledSeries(
        values=resampled,
        timestamp_ns=timestamps,
        rate=rate,
        source_sample_count=n,
    )


def resample_128_hz_to_50_hz(
    values: np.ndarray,
    *,
    start_timestamp_ns: int,
) -> ResampledSeries:
    """Resample 128 Hz IMU to 50 Hz using exactly ``up=25`` and ``down=64``."""

    series = resample_polyphase(
        values, source_hz=128, target_hz=50, start_timestamp_ns=start_timestamp_ns
    )
    # Raised, not asserted: `python -O` strips `assert`, and this is the one
    # statement binding this function's name to the 25/64 factor it promises.
    if series.rate.up != 25 or series.rate.down != 64:
        raise ResamplingError("resampling_rate_contract_violated")
    return series


__all__ = [
    "RESAMPLING_SCHEMA_VERSION",
    "RationalRate",
    "ResampledSeries",
    "ResamplingError",
    "resample_polyphase",
    "resample_128_hz_to_50_hz",
]


In [ ]:
# Cell 5 -- preprocess.py (SisFall)
%%writefile /content/repo/ml/training/preprocess.py
"""
preprocess.py

Tien xu ly bo du lieu SisFall cho bai toan Fall Detection (v2):
1. Doc tat ca file .txt tu SisFall_dataset
2. Chuyen doi don vi (raw -> G, raw -> deg/s)
3. Downsample tu 200Hz xuong 50Hz
4. Them kenh SVM (Signal Vector Magnitude) lam kenh thu 7
5. Sliding window: 2 giay (100 mau), overlap 50%
6. Gan nhan: D* = NORMAL (0), F* = FALL (1)
7. Chia train/test theo Subject (tranh data leakage)
8. Luu ket qua vao ml/data/processed/
"""

import os
import sys
import numpy as np
from pathlib import Path
import time

# ===========================================================================
# CAU HINH
# ===========================================================================
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
DATASET_DIR = PROJECT_ROOT / "ml" / "data" / "raw" / "SisFall_dataset"
OUTPUT_DIR = PROJECT_ROOT / "ml" / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# He so chuyen doi
ADXL345_SCALE = 256.0       # raw -> G
ITG3200_SCALE = 14.375      # raw -> deg/s

# Tham so xu ly
ORIGINAL_RATE = 200          # Hz (tan so goc cua SisFall)
TARGET_RATE = 50             # Hz (tan so muc tieu, giong ESP32)
DOWNSAMPLE_FACTOR = ORIGINAL_RATE // TARGET_RATE  # = 4

WINDOW_SIZE = 100            # 100 mau = 2 giay tai 50Hz (bat tron 3 pha cua cu nga)
WINDOW_STEP = 50             # Buoc nhay 50 mau = overlap 50%

# Chia train/test theo Subject
# Train: SA01-SA18 + SE01-SE12 (30 nguoi)
# Test:  SA19-SA23 + SE13-SE15 (8 nguoi)
TEST_SUBJECTS = {'SA19', 'SA20', 'SA21', 'SA22', 'SA23', 'SE13', 'SE14', 'SE15'}


# ===========================================================================
# HAM XU LY
# ===========================================================================

def load_sisfall_file(filepath):
    """
    Doc 1 file .txt cua SisFall, lay 6 cot dau (ADXL345 + ITG3200),
    chuyen doi don vi, va them kenh SVM.
    
    Returns:
        numpy array shape (N, 7): [ax, ay, az, gx, gy, gz, svm]
        ax, ay, az: don vi G
        gx, gy, gz: don vi deg/s
    """
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip().rstrip(';').strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) >= 6:
                try:
                    values = [int(p.strip()) for p in parts[:6]]
                    rows.append(values)
                except ValueError:
                    continue

    if len(rows) == 0:
        return None

    data = np.array(rows, dtype=np.float64)

    # Chuyen doi don vi
    data[:, 0:3] /= ADXL345_SCALE   # Accelerometer -> G
    data[:, 3:6] /= ITG3200_SCALE   # Gyroscope -> deg/s

    # Tinh SVM (Signal Vector Magnitude) va them lam kenh thu 7
    ax, ay, az = data[:, 0], data[:, 1], data[:, 2]
    svm = np.sqrt(ax**2 + ay**2 + az**2).reshape(-1, 1)
    data = np.hstack([data, svm])  # shape: (N, 7)

    return data


def downsample(data, factor):
    """Giam tan so lay mau bang cach lay moi 'factor' dong."""
    return data[::factor]


def create_windows(data, window_size, step):
    """
    Cat chuoi du lieu thanh cac cua so (windows).
    
    Args:
        data: numpy array (N, 6)
        window_size: so mau moi cua so
        step: buoc nhay

    Returns:
        numpy array (num_windows, window_size, 6)
    """
    windows = []
    for start in range(0, len(data) - window_size + 1, step):
        window = data[start:start + window_size]
        windows.append(window)

    if len(windows) == 0:
        return np.array([]).reshape(0, window_size, data.shape[1])

    return np.array(windows)


def get_label_from_filename(filename):
    """
    Xac dinh nhan tu ten file.
    D* -> NORMAL (0)
    F* -> FALL (1)
    """
    name = Path(filename).stem  # e.g., D01_SA01_R01
    if name.startswith('D'):
        return 0  # NORMAL
    elif name.startswith('F'):
        return 1  # FALL
    else:
        return -1  # Khong xac dinh


def get_subject_from_filename(filename):
    """Lay Subject ID tu ten file. VD: D01_SA01_R01 -> SA01"""
    name = Path(filename).stem
    parts = name.split('_')
    if len(parts) >= 2:
        return parts[1]  # SA01, SE06, ...
    return None


def process_all_files():
    """
    Xu ly toan bo dataset:
    1. Doc tung file
    2. Downsample
    3. Tao sliding windows
    4. Gan nhan
    5. Chia train/test theo Subject
    """
    train_windows = []
    train_labels = []
    test_windows = []
    test_labels = []

    total_files = 0
    skipped_files = 0
    
    # Dem tong so file truoc
    all_files = []
    for subject_dir in sorted(DATASET_DIR.iterdir()):
        if not subject_dir.is_dir():
            continue
        for filepath in sorted(subject_dir.glob("*.txt")):
            if filepath.name.lower() in ('readme.txt', 'desktop.ini'):
                continue
            all_files.append(filepath)

    print(f"\n  Tong so file can xu ly: {len(all_files)}")
    print(f"  Downsample: {ORIGINAL_RATE}Hz -> {TARGET_RATE}Hz (he so {DOWNSAMPLE_FACTOR})")
    print(f"  Window: {WINDOW_SIZE} mau ({WINDOW_SIZE/TARGET_RATE:.1f}s), overlap {(1 - WINDOW_STEP/WINDOW_SIZE)*100:.0f}%")
    print(f"  Test subjects: {sorted(TEST_SUBJECTS)}")
    print()

    start_time = time.time()

    for i, filepath in enumerate(all_files):
        total_files += 1

        # Hien thi tien do
        if (i + 1) % 100 == 0 or (i + 1) == len(all_files):
            elapsed = time.time() - start_time
            pct = (i + 1) / len(all_files) * 100
            print(f"  [{i+1}/{len(all_files)}] ({pct:.0f}%) - {elapsed:.1f}s", end='\r')

        # Doc file
        data = load_sisfall_file(filepath)
        if data is None or len(data) < DOWNSAMPLE_FACTOR * WINDOW_SIZE:
            skipped_files += 1
            continue

        # Downsample
        data = downsample(data, DOWNSAMPLE_FACTOR)

        # Tao windows
        windows = create_windows(data, WINDOW_SIZE, WINDOW_STEP)
        if len(windows) == 0:
            skipped_files += 1
            continue

        # Gan nhan
        label = get_label_from_filename(filepath.name)
        if label == -1:
            skipped_files += 1
            continue

        labels = np.full(len(windows), label)

        # Chia vao train hoac test dua tren Subject
        subject = get_subject_from_filename(filepath.name)
        if subject in TEST_SUBJECTS:
            test_windows.append(windows)
            test_labels.append(labels)
        else:
            train_windows.append(windows)
            train_labels.append(labels)

    print()  # Xuong dong sau progress bar
    elapsed = time.time() - start_time
    print(f"\n  Thoi gian xu ly: {elapsed:.1f} giay")
    print(f"  File da xu ly  : {total_files - skipped_files}/{total_files}")
    print(f"  File bo qua    : {skipped_files}")

    # Gom tat ca windows lai
    X_train = np.concatenate(train_windows, axis=0)
    y_train = np.concatenate(train_labels, axis=0)
    X_test = np.concatenate(test_windows, axis=0)
    y_test = np.concatenate(test_labels, axis=0)

    return X_train, y_train, X_test, y_test


def main():
    print("=" * 60)
    print("  TIEN XU LY DU LIEU SISFALL")
    print("=" * 60)

    # Kiem tra dataset ton tai
    if not DATASET_DIR.exists():
        print(f"\n  LOI: Khong tim thay thu muc dataset tai:")
        print(f"  {DATASET_DIR}")
        print(f"  Hay giai nen SisFall vao thu muc ml/data/raw/SisFall_dataset/")
        sys.exit(1)

    # Xu ly
    X_train, y_train, X_test, y_test = process_all_files()

    # Thong ke ket qua
    print("\n" + "-" * 60)
    print("  KET QUA TIEN XU LY")
    print("-" * 60)
    print(f"\n  TRAIN SET:")
    print(f"    Tong so windows : {len(X_train)}")
    print(f"    NORMAL (0)      : {np.sum(y_train == 0)}")
    print(f"    FALL (1)        : {np.sum(y_train == 1)}")
    print(f"    Shape           : {X_train.shape}")
    print(f"    (= {X_train.shape[0]} windows x {X_train.shape[1]} mau x {X_train.shape[2]} kenh)")

    print(f"\n  TEST SET:")
    print(f"    Tong so windows : {len(X_test)}")
    print(f"    NORMAL (0)      : {np.sum(y_test == 0)}")
    print(f"    FALL (1)        : {np.sum(y_test == 1)}")
    print(f"    Shape           : {X_test.shape}")

    # Luu file
    print(f"\n  Dang luu vao: {OUTPUT_DIR}")
    np.savez_compressed(OUTPUT_DIR / "X_train.npz", data=X_train)
    np.savez_compressed(OUTPUT_DIR / "y_train.npz", data=y_train)
    np.savez_compressed(OUTPUT_DIR / "X_test.npz", data=X_test)
    np.savez_compressed(OUTPUT_DIR / "y_test.npz", data=y_test)

    # Kiem tra dung luong file
    total_size = 0
    for f in OUTPUT_DIR.glob("*.npz"):
        size = f.stat().st_size
        total_size += size
        print(f"    {f.name}: {size / 1024 / 1024:.1f} MB")
    print(f"    Tong: {total_size / 1024 / 1024:.1f} MB")

    print("\n" + "=" * 60)
    print("  HOAN THANH TIEN XU LY!")
    print("=" * 60)


if __name__ == "__main__":
    main()


In [ ]:
# Cell 6 -- preprocess_kfall.py (KFall)
%%writefile /content/repo/ml/training/preprocess_kfall.py
"""
preprocess_kfall.py

Tich hop KFall dataset vao pipeline 3-class (NORMAL / INSTABILITY / FALL).

KFall dac diem:
  - 32 subjects (SA06-SA38, missing SA34)
  - 100 Hz (can downsample xuong 50 Hz)
  - Gyroscope don vi rad/s (can chuyen sang deg/s)
  - Task T01-T19 = ADL (NORMAL), T20-T34 = FALL
  - Label xlsx co Fall_onset_frame va Fall_impact_frame (o 100 Hz)

Output: them window moi vao X_train.npz, y_train.npz
        Test set giu nguyen de dam bao fairness.

Chay tu thu muc goc project:
  python ml/training/preprocess_kfall.py
"""

from __future__ import annotations

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ===========================================================================
# THAM SO
# ===========================================================================
WINDOW_SIZE  = 100          # 2 giay o 50 Hz
WINDOW_STEP  = 50           # overlap 50% (khop voi preprocess.py)
SOURCE_HZ    = 100          # KFall sampling rate
TARGET_HZ    = 50           # pipeline sampling rate
DOWNSAMPLE   = SOURCE_HZ // TARGET_HZ   # = 2, lay moi sample thu 2

# KFall gyroscope da o don vi deg/s (LPMS-B2 output), KHONG can convert

# Task ID: T01-T19 = ADL (NORMAL=0), T20-T34 = FALL (FALL=2)
ADL_TASK_MAX  = 19
FALL_TASK_MIN = 20

# Bao nhieu frame (o 50Hz) sau impact thi ket thuc vung FALL
FALL_POST_IMPACT = 50       # 1 giay

# ===========================================================================
# DUONG DAN
# ===========================================================================
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
KFALL_DIR    = PROJECT_ROOT / "ml" / "data" / "raw" / "KFALL_dataset"
DATA_DIR     = PROJECT_ROOT / "ml" / "data" / "processed_3class"
DATA_DIR.mkdir(parents=True, exist_ok=True)


# ===========================================================================
# HELPERS
# ===========================================================================

def load_csv(path: Path) -> np.ndarray:
    """
    Doc CSV KFall, downsample 100->50 Hz, convert gyro rad/s->deg/s,
    tinh SVM, tra ve array shape (N, 7): [ax, ay, az, gx, gy, gz, svm]
    """
    df = pd.read_csv(path)
    # Downsample: lay moi sample thu DOWNSAMPLE
    df = df.iloc[::DOWNSAMPLE].reset_index(drop=True)

    ax = df["AccX"].values.astype(np.float32)
    ay = df["AccY"].values.astype(np.float32)
    az = df["AccZ"].values.astype(np.float32)
    gx = df["GyrX"].values.astype(np.float32)
    gy = df["GyrY"].values.astype(np.float32)
    gz = df["GyrZ"].values.astype(np.float32)
    svm = np.sqrt(ax**2 + ay**2 + az**2).astype(np.float32)

    return np.stack([ax, ay, az, gx, gy, gz, svm], axis=1)   # (N, 7)


def parse_label_xlsx(xlsx_path: Path) -> dict[tuple[int, int], tuple[int, int]]:
    """
    Doc label file, tra ve dict: {(task_id, trial_id): (onset_50hz, impact_50hz)}
    """
    lb = pd.read_excel(xlsx_path)
    result = {}
    current_task = None
    current_task_id = None

    for _, row in lb.iterrows():
        task_str = row["Task Code (Task ID)"]
        if pd.notna(task_str):
            m = re.search(r"\((\d+)\)", str(task_str))
            current_task_id = int(m.group(1)) if m else None
            current_task    = task_str

        trial    = int(row["Trial ID"])
        onset_100 = int(row["Fall_onset_frame"])
        impact_100 = int(row["Fall_impact_frame"])

        # Chuyen frame 100Hz -> 50Hz
        onset_50  = onset_100  // DOWNSAMPLE
        impact_50 = impact_100 // DOWNSAMPLE

        if current_task_id is not None:
            result[(current_task_id, trial)] = (onset_50, impact_50)

    return result


def sliding_windows(data: np.ndarray, label: int) -> tuple[list, list]:
    """Sliding window tren toan bo file, tra ve (windows, labels)."""
    X, y = [], []
    for start in range(0, len(data) - WINDOW_SIZE + 1, WINDOW_STEP):
        X.append(data[start : start + WINDOW_SIZE])
        y.append(label)
    return X, y


def fall_windows(data: np.ndarray,
                 onset_50: int,
                 impact_50: int) -> tuple[list, list]:
    """
    Trich xuat windows tu file FALL:
      - Truoc onset: NORMAL
      - Tu onset den impact + FALL_POST_IMPACT: FALL
      - Sau do: NORMAL (giai doan recovery)
    """
    fall_end = min(impact_50 + FALL_POST_IMPACT, len(data))

    X_norm, y_norm = [], []
    X_fall, y_fall = [], []

    for start in range(0, len(data) - WINDOW_SIZE + 1, WINDOW_STEP):
        end = start + WINDOW_SIZE
        win = data[start:end]

        # Window trung voi vung fall?
        overlap_start = max(start, onset_50)
        overlap_end   = min(end,   fall_end)

        if overlap_end > overlap_start:
            X_fall.append(win)
            y_fall.append(2)   # FALL
        elif end <= onset_50:
            X_norm.append(win)
            y_norm.append(0)   # NORMAL (truoc fall)

    return X_norm + X_fall, y_norm + y_fall


# ===========================================================================
# MAIN PROCESSING
# ===========================================================================

def process_subject(subject_dir: Path) -> tuple[list, list]:
    """Xu ly 1 subject, tra ve (X_list, y_list)."""
    subj_id = subject_dir.name   # e.g. SA06
    num = subj_id[2:]            # e.g. 06

    # Tim label file
    label_path = subject_dir.parent / f"{subj_id}_label.xlsx"
    if not label_path.exists():
        print(f"  [WARN] label file not found for {subj_id}, skip")
        return [], []

    labels = parse_label_xlsx(label_path)

    X_all, y_all = [], []

    for csv_path in sorted(subject_dir.glob("*.csv")):
        m = re.match(r"S(\d+)T(\d+)R(\d+)", csv_path.stem)
        if not m:
            continue
        task_id  = int(m.group(2))
        trial_id = int(m.group(3))

        try:
            data = load_csv(csv_path)
        except Exception as e:
            print(f"  [WARN] {csv_path.name}: {e}")
            continue

        if task_id <= ADL_TASK_MAX:
            # ADL -> NORMAL windows
            Xw, yw = sliding_windows(data, label=0)
        elif task_id >= FALL_TASK_MIN:
            # FALL file: dung label de phan loai
            key = (task_id, trial_id)
            if key not in labels:
                # Khong co label -> bo qua
                continue
            onset_50, impact_50 = labels[key]
            if onset_50 >= len(data) or impact_50 >= len(data):
                continue
            Xw, yw = fall_windows(data, onset_50, impact_50)
        else:
            continue

        X_all.extend(Xw)
        y_all.extend(yw)

    return X_all, y_all


def main():
    print("=" * 60)
    print("  PREPROCESS KFALL DATASET")
    print(f"  Source: {KFALL_DIR}")
    print("=" * 60)

    subjects = sorted([d for d in KFALL_DIR.iterdir() if d.is_dir()])
    print(f"  Found {len(subjects)} subjects")

    X_new, y_new = [], []

    for subj in subjects:
        Xs, ys = process_subject(subj)
        if Xs:
            X_new.extend(Xs)
            y_new.extend(ys)
            n0 = sum(1 for v in ys if v == 0)
            n2 = sum(1 for v in ys if v == 2)
            print(f"  {subj.name}: {len(Xs):5d} windows  (NORMAL={n0}, FALL={n2})")

    X_new = np.array(X_new, dtype=np.float32)
    y_new = np.array(y_new, dtype=np.int32)

    print(f"\n  KFall windows: {len(X_new):,}")
    for cls, name in [(0, "NORMAL"), (2, "FALL")]:
        print(f"    {name:15s}: {int((y_new==cls).sum()):6,}")

    # Load existing train set
    print("\n  Loading existing train set...")
    X_old = np.load(DATA_DIR / "X_train.npz")["data"]
    y_old = np.load(DATA_DIR / "y_train.npz")["data"].astype(np.int32)
    print(f"  Existing: {len(X_old):,} windows")

    # Merge
    X_merged = np.concatenate([X_old, X_new], axis=0)
    y_merged = np.concatenate([y_old, y_new], axis=0)

    # Shuffle
    rng  = np.random.default_rng(42)
    perm = rng.permutation(len(X_merged))
    X_merged = X_merged[perm]
    y_merged = y_merged[perm]

    print(f"\n  Merged train set: {len(X_merged):,} windows")
    for cls, name in [(0, "NORMAL"), (1, "INSTABILITY"), (2, "FALL")]:
        print(f"    {name:15s}: {int((y_merged==cls).sum()):6,}")

    # Backup old files
    bak_X = DATA_DIR / "X_train_pre_kfall.npz"
    bak_y = DATA_DIR / "y_train_pre_kfall.npz"
    if not bak_X.exists():
        import shutil
        shutil.copy(DATA_DIR / "X_train.npz", bak_X)
        shutil.copy(DATA_DIR / "y_train.npz", bak_y)
        print(f"\n  Backup saved: X_train_pre_kfall.npz, y_train_pre_kfall.npz")

    # Save merged
    np.savez_compressed(DATA_DIR / "X_train.npz", data=X_merged)
    np.savez_compressed(DATA_DIR / "y_train.npz", data=y_merged)
    print(f"  Saved: X_train.npz ({(DATA_DIR/'X_train.npz').stat().st_size/1024/1024:.1f} MB)")
    print(f"  Saved: y_train.npz")

    print("\n  Buoc tiep theo:")
    print("  -> Upload X_train.npz moi len Google Drive")
    print("  -> Chay lai train_models_colab.ipynb (Cell 9: CNN+BiLSTM+Attn)")
    print("  -> Sau khi train xong chay lai baseline_comparison.py")


if __name__ == "__main__":
    main()


In [ ]:
# Cell 7 -- preprocess_vestibular.py (VestibularDataset)
%%writefile /content/repo/ml/training/preprocess_vestibular.py
"""
preprocess_vestibular.py

Tien xu ly bo du lieu Vestibular (BV/UV/HS) va ket hop voi SisFall
de tao dataset 3 class cho bai toan phat hien tien dinh:

  NORMAL (0)                  -- hoat dong binh thuong
  VESTIBULAR_INSTABILITY (1)  -- mat on dinh van dong dac trung tien dinh
  FALL (2)                    -- su kien te nga (tu SisFall)

LUU Y (M2 narrow repair):
  * Truoc day module lay mau thua 128Hz -> 64Hz (DOWNSAMPLE_FACTOR = 128 // 50).
    Bug nay lam mot cua so 100 mau chi dai 1.5625 s thay vi 2 s. Da thay bang
    resample huu ty chong chong (anti-aliased polyphase) up=25 / down=64 qua
    ml.common.resampling.
  * Nhan NORMAL / VESTIBULAR_INSTABILITY o day KHONG phai nhan canonical M1 va
    KHONG du dieu kien lam bang chung INSTABILITY cho M4/M5. Manifest ghi ro co
    legacy_label_semantics_ineligible_for_instability = True.
  * Import module khong con tao thu muc output; chi ham main() moi ghi file.
"""

import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import FrozenSet, Mapping, Optional, Tuple

import numpy as np
import scipy

from ml.common.resampling import (
    RESAMPLING_SCHEMA_VERSION,
    resample_128_hz_to_50_hz,
)

# ===========================================================================
# CAU HINH
# ===========================================================================
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent

VEST_DIR = PROJECT_ROOT / "ml" / "data" / "raw" / "vestibular_dataset"
SISFALL_DIR = PROJECT_ROOT / "ml" / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "ml" / "data" / "processed_3class"

# Sampling
ORIGINAL_RATE = 128  # Hz cua vestibular dataset
TARGET_RATE = 50     # Hz cua ESP32 va SisFall pipeline
_RESAMPLE_UP = 25
_RESAMPLE_DOWN = 64

# Window
WINDOW_SIZE = 100  # 100 mau = 2 giay tai 50Hz
WINDOW_STEP = 50   # overlap 50%

# So mau nguon toi thieu de resample ra du 100 mau 50Hz:
# ceil(n * 25 / 64) >= 100  =>  n >= 256
MIN_SOURCE_SAMPLES_FOR_WINDOW = 256

# Labels (legacy, KHONG phai canonical M1)
LABEL_NORMAL = 0
LABEL_INSTAB = 1
LABEL_FALL = 2

# ===========================================================================
# PHAN LOAI ACTIVITY -> LABEL
# ===========================================================================

INSTABILITY_ACTIVITIES = {
    "Darkness walk",
    "Wood beam",
    "Uneven ground",
    "Stairs",
    "Stepladder",
    "Inclined plane",
    "Bus",
    "Pants",
    "Shoes",
    "Tray",
    "Heavy load",
}

NORMAL_ACTIVITIES_PATIENT = {
    "Walk",
    "Bed",
    "Sorting",
    "Picture recognition",
}

# ===========================================================================
# TRAIN / TEST SPLIT (theo participant ID de tranh leakage)
# ===========================================================================

BV_TEST = {f"Participant_{i:02d}_BV" for i in range(15, 20)}
UV_TEST = {f"Participant_{i:02d}_UV" for i in range(35, 40)}
HS_TEST = {f"Participant_{i:02d}_HS" for i in range(55, 60)}
TEST_PARTICIPANTS = BV_TEST | UV_TEST | HS_TEST

BV_TRAIN = {f"Participant_{i:02d}_BV" for i in range(1, 15)}
UV_TRAIN = {f"Participant_{i:02d}_UV" for i in range(20, 35)}
HS_TRAIN = {f"Participant_{i:02d}_HS" for i in range(40, 55)}
TRAIN_PARTICIPANTS = BV_TRAIN | UV_TRAIN | HS_TRAIN


# ===========================================================================
# RESAMPLING (M2 narrow repair)
# ===========================================================================

def resample_to_target_rate(data: np.ndarray) -> np.ndarray:
    """128 Hz -> 50 Hz bang anti-aliased polyphase up=25 / down=64.

    Nhan mang (N, C) float; tra ve (ceil(N * 25 / 64), C) cung dtype.
    """
    array = np.asarray(data)
    if array.ndim != 2:
        raise ValueError("resample_to_target_rate_expects_2d")
    if array.dtype.kind != "f":
        array = array.astype(np.float64)
    series = resample_128_hz_to_50_hz(array, start_timestamp_ns=0)
    return series.values


@dataclass(frozen=True)
class ObservedSplit:
    """Participants actually routed, as opposed to the declared sets.

    ``unlisted`` names folders present in neither declared set. The routing rule
    is ``not in TEST_PARTICIPANTS``, so those are trained on; recording them is
    what keeps the provenance record honest about who was trained on.
    """

    train: FrozenSet[str]
    test: FrozenSet[str]
    unlisted: Tuple[str, ...] = ()


def build_legacy_source_split_manifest(
    observed: Optional[ObservedSplit] = None,
) -> Mapping[str, object]:
    """Manifest hai chieu legacy: KHONG phai canonical M1 fold."""
    # Reported only when a real pass happened. Absent rather than empty, because
    # an empty `observed_unlisted_participants` would read as "no unlisted
    # folders found" when in fact nothing was inspected.
    observed_fields: Mapping[str, object] = (
        {}
        if observed is None
        else {
            "observed_train_participants": sorted(observed.train),
            "observed_held_out_participants": sorted(observed.test),
            "observed_unlisted_participants": list(observed.unlisted),
        }
    )
    return {
        **observed_fields,
        "schema_version": RESAMPLING_SCHEMA_VERSION,
        "source_rate_hz": ORIGINAL_RATE,
        "target_rate_hz": TARGET_RATE,
        "resample_up": _RESAMPLE_UP,
        "resample_down": _RESAMPLE_DOWN,
        "scipy_version": scipy.__version__,
        "train_participants": sorted(TRAIN_PARTICIPANTS),
        "held_out_participants": sorted(TEST_PARTICIPANTS),
        "participant_group_ids": sorted(TRAIN_PARTICIPANTS | TEST_PARTICIPANTS),
        "canonical_m1_folds": False,
        "eligibility": "legacy_public_pretraining_only",
        "legacy_label_semantics_ineligible_for_instability": True,
    }


# ===========================================================================
# HAM XU LY
# ===========================================================================

def get_participant_group(folder_name):
    if folder_name.endswith("_BV"):
        return "BV"
    if folder_name.endswith("_UV"):
        return "UV"
    if folder_name.endswith("_HS"):
        return "HS"
    return None


def get_activity_label(group, activity_name):
    if group == "HS":
        return LABEL_NORMAL
    if group in ("BV", "UV"):
        if activity_name in INSTABILITY_ACTIVITIES:
            return LABEL_INSTAB
        if activity_name in NORMAL_ACTIVITIES_PATIENT:
            return LABEL_NORMAL
        return None
    return None


def load_vest_file(csv_path):
    """Doc 1 file CSV vestibular, lay 6 kenh TR (Trunk sensor) + kenh SVM."""
    import pandas as pd  # lazy: pandas khong nam trong M2 pretrain lock

    df = pd.read_csv(
        csv_path,
        usecols=[
            "Acc_x_g_TR", "Acc_y_g_TR", "Acc_z_g_TR",
            "Gyr_x_deg/s_TR", "Gyr_y_deg/s_TR", "Gyr_z_deg/s_TR",
        ],
    )
    df = df.dropna()
    if len(df) == 0:
        return None

    # Six axes only. The SVM magnitude channel is appended *after* resampling by
    # `add_svm_channel`, because a magnitude is nonlinear: low-pass filtering
    # sqrt(ax^2+ay^2+az^2) does not equal the magnitude of the filtered axes, so
    # computing it here and then polyphase-filtering it with the rest left
    # channel 7 inconsistent with channels 1-3 and with the SisFall convention
    # these windows are concatenated with.
    return df.values.astype(np.float64)  # (N, 6)


def add_svm_channel(data: np.ndarray) -> np.ndarray:
    """Append the signal vector magnitude of the three accelerometer axes.

    Must run on the series at its final rate. Kept separate from loading so the
    identity ``channel 7 == magnitude(channels 1..3)`` holds in the output.
    """

    ax, ay, az = data[:, 0], data[:, 1], data[:, 2]
    svm = np.sqrt(ax ** 2 + ay ** 2 + az ** 2).reshape(-1, 1)
    return np.hstack([data, svm])  # (N, 7)


def create_windows(data, window_size, step):
    windows = []
    for start in range(0, len(data) - window_size + 1, step):
        windows.append(data[start:start + window_size])
    if not windows:
        return np.empty((0, window_size, data.shape[1]))
    return np.array(windows)


# ===========================================================================
# XU LY VESTIBULAR DATASET
# ===========================================================================

def process_participant(p_dir: Path):
    """Process one participant folder in isolation.

    Returns ``(windows, labels, file_count, skipped_count)`` for exactly this
    participant, with no train/test routing decision -- callers decide that.
    ``windows``/``labels`` are ``None`` when the folder is not a recognised
    group or has no ``.csv`` subdirectory (not merely "zero usable files").
    """
    group = get_participant_group(p_dir.name)
    if group is None:
        return None, None, 0, 0
    csv_dir = p_dir / ".csv"
    if not csv_dir.exists():
        return None, None, 0, 0

    X_list, y_list = [], []
    file_count = 0
    skipped = 0

    for csv_file in sorted(csv_dir.glob("IMU_*.csv")):
        activity = csv_file.stem.replace("IMU_", "")
        label = get_activity_label(group, activity)
        if label is None:
            skipped += 1
            continue

        data = load_vest_file(csv_file)
        if data is None or len(data) < MIN_SOURCE_SAMPLES_FOR_WINDOW:
            skipped += 1
            continue

        # Resample the six measured axes, then derive SVM at the target rate.
        data = resample_to_target_rate(data)
        data = add_svm_channel(data)
        windows = create_windows(data, WINDOW_SIZE, WINDOW_STEP)
        if len(windows) == 0:
            skipped += 1
            continue

        X_list.append(windows)
        y_list.append(np.full(len(windows), label, dtype=np.int32))
        file_count += 1

    if not X_list:
        return (
            np.empty((0, WINDOW_SIZE, 7)),
            np.empty((0,), dtype=np.int32),
            file_count,
            skipped,
        )
    return (
        np.concatenate(X_list, axis=0),
        np.concatenate(y_list, axis=0),
        file_count,
        skipped,
    )


def process_vestibular():
    train_X, train_y = [], []
    test_X, test_y = [], []
    # Which participants were *actually* routed where. The manifest used to report
    # the hardcoded `TRAIN_PARTICIPANTS` set while the routing decision is
    # `not in TEST_PARTICIPANTS`, so a folder in neither list -- a new cohort, a
    # renamed directory -- silently joined TRAIN while being absent from the
    # provenance record.
    observed_train: set = set()
    observed_test: set = set()

    participants = sorted(VEST_DIR.iterdir())
    total_files = 0
    skipped = 0

    print(f"  Tim thay {len(participants)} participant folders")

    for p_dir in participants:
        if not p_dir.is_dir():
            continue
        windows, labels, file_count, file_skipped = process_participant(p_dir)
        skipped += file_skipped
        if windows is None or len(windows) == 0:
            continue

        total_files += file_count
        is_test = p_dir.name in TEST_PARTICIPANTS
        if is_test:
            test_X.append(windows)
            test_y.append(labels)
            observed_test.add(p_dir.name)
        else:
            train_X.append(windows)
            train_y.append(labels)
            observed_train.add(p_dir.name)

    print(f"  Files xu ly thanh cong : {total_files}")
    print(f"  Files bo qua           : {skipped}")

    X_tr = np.concatenate(train_X, axis=0) if train_X else np.empty((0, WINDOW_SIZE, 7))
    y_tr = np.concatenate(train_y, axis=0) if train_y else np.empty((0,), dtype=np.int32)
    X_te = np.concatenate(test_X, axis=0) if test_X else np.empty((0, WINDOW_SIZE, 7))
    y_te = np.concatenate(test_y, axis=0) if test_y else np.empty((0,), dtype=np.int32)
    unlisted = sorted((observed_train | observed_test) - (TRAIN_PARTICIPANTS | TEST_PARTICIPANTS))
    if unlisted:
        print(f"  CANH BAO: participant khong co trong danh sach khai bao: {unlisted}")
    return X_tr, y_tr, X_te, y_te, ObservedSplit(
        train=frozenset(observed_train),
        test=frozenset(observed_test),
        unlisted=tuple(unlisted),
    )


def load_sisfall_processed():
    X_tr = np.load(SISFALL_DIR / "X_train.npz")["data"]
    y_tr = np.load(SISFALL_DIR / "y_train.npz")["data"].astype(np.int32)
    X_te = np.load(SISFALL_DIR / "X_test.npz")["data"]
    y_te = np.load(SISFALL_DIR / "y_test.npz")["data"].astype(np.int32)
    y_tr[y_tr == 1] = LABEL_FALL
    y_te[y_te == 1] = LABEL_FALL
    return X_tr, y_tr, X_te, y_te


# ===========================================================================
# MAIN
# ===========================================================================

def _write_provenance_manifest(observed: Optional[ObservedSplit] = None) -> None:
    import json

    manifest = dict(build_legacy_source_split_manifest(observed))
    manifest["window_size"] = WINDOW_SIZE
    manifest["window_step"] = WINDOW_STEP
    manifest["min_source_samples_for_window"] = MIN_SOURCE_SAMPLES_FOR_WINDOW
    (OUTPUT_DIR / "legacy_vestibular_provenance.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )


def print_stats(label, y):
    print(f"  {label}:")
    print(f"    NORMAL (0)                 : {np.sum(y == 0)}")
    print(f"    VESTIBULAR_INSTABILITY (1) : {np.sum(y == 1)}")
    print(f"    FALL (2)                   : {np.sum(y == 2)}")
    print(f"    Tong                       : {len(y)}")


def main():
    print("=" * 60)
    print("  TIEN XU LY VESTIBULAR + SISFALL (3 CLASS, LEGACY)")
    print("=" * 60)

    if not VEST_DIR.exists():
        print(f"\n  LOI: Khong tim thay {VEST_DIR}")
        sys.exit(1)
    if not SISFALL_DIR.exists():
        print(f"\n  LOI: Khong tim thay {SISFALL_DIR}")
        sys.exit(1)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("\n[1/3] Xu ly Vestibular dataset...")
    t0 = time.time()
    vX_tr, vy_tr, vX_te, vy_te, observed_split = process_vestibular()
    print(f"  Thoi gian: {time.time() - t0:.1f}s")

    print("\n[2/3] Load SisFall processed (remap FALL 1->2)...")
    sX_tr, sy_tr, sX_te, sy_te = load_sisfall_processed()

    print("\n[3/3] Merge va luu...")
    X_train = np.concatenate([vX_tr, sX_tr], axis=0)
    y_train = np.concatenate([vy_tr, sy_tr], axis=0)
    X_test = np.concatenate([vX_te, sX_te], axis=0)
    y_test = np.concatenate([vy_te, sy_te], axis=0)

    print_stats("TRAIN", y_train)
    print_stats("TEST ", y_test)

    np.savez_compressed(OUTPUT_DIR / "X_train.npz", data=X_train)
    np.savez_compressed(OUTPUT_DIR / "y_train.npz", data=y_train)
    np.savez_compressed(OUTPUT_DIR / "X_test.npz", data=X_test)
    np.savez_compressed(OUTPUT_DIR / "y_test.npz", data=y_test)
    _write_provenance_manifest(observed_split)

    print(f"\n  Da luu vao: {OUTPUT_DIR}")
    print("  Legacy label semantics: KHONG canonical, KHONG du dieu kien INSTABILITY.")


if __name__ == "__main__":
    main()


In [ ]:
# Cell 8 -- build_subject_independent_dataset.py (orchestrator)
%%writefile /content/repo/ml/training/build_subject_independent_dataset.py
"""
build_subject_independent_dataset.py

Rebuilds the 3-class (NORMAL / INSTABILITY / FALL) dataset with a genuine
subject-independent TRAIN / VAL / TEST split across all three source
datasets (SisFall, KFall, VestibularDataset). This fixes three problems the
2026-09 paper audit found in the existing `preprocess*.py` -> Colab pipeline:

  1. KFall was previously merged into TRAIN only ("Test set giu nguyen de
     dam bao fairness" in the old `preprocess_kfall.py`) -- the reported
     TEST set never contained a single KFall window, so no reported metric
     ever measured generalisation to KFall-style recordings.
  2. There was no genuine held-out VALIDATION split at all: the Colab
     notebook's `model.fit(..., validation_data=(X_test_n, y_te_cat))` used
     the TEST set itself as the early-stopping signal, and the committed
     threshold-sweep script also ran on the test set. Both are leakage --
     model selection and threshold selection must never see test data.
  3. INSTABILITY windows come exclusively from VestibularDataset, never from
     KFall (KFall's own label routing is NORMAL/FALL only). This script
     keeps that composition explicit and auditable in the manifest instead
     of letting later documentation imply otherwise.

This script does NOT change any per-file feature extraction, resampling, or
windowing logic. It imports and reuses, unchanged, the existing per-subject
loaders already fixed by the M2 vestibular-resampling repair:
  - `preprocess.py`            (SisFall: 200 Hz -> 50 Hz)
  - `preprocess_kfall.py`      (KFall: 100 Hz -> 50 Hz)
  - `preprocess_vestibular.py` (VestibularDataset: 128 Hz -> 50 Hz, already
    fixed to a true anti-aliased polyphase resample in commit cc7b2c7)
It only replaces *how subjects are assigned to splits*.

Run from the project root, after all three raw datasets are unpacked under
`ml/data/raw/` exactly as the three underlying scripts expect
(`SisFall_dataset/`, `KFALL_dataset/`, `vestibular_dataset/`):

  python -m ml.training.build_subject_independent_dataset

Output, under `ml/data/processed_3class_v2/`:
  X_train.npz / y_train.npz
  X_val.npz   / y_val.npz
  X_test.npz  / y_test.npz
  dataset_manifest.json   -- fills the paper's Methods dataset table
                             programmatically; never hand-type these numbers.

Upload the whole `processed_3class_v2/` directory to Google Drive for the
Colab training notebook (`train_models_colab_v2.ipynb`).
"""

from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np

from ml.training import preprocess as sisfall_pp
from ml.training import preprocess_kfall as kfall_pp
from ml.training import preprocess_vestibular as vest_pp

# ===========================================================================
# CONFIG
# ===========================================================================
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
OUTPUT_DIR = PROJECT_ROOT / "ml" / "data" / "processed_3class_v2"

SPLIT_SEED = 42
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
# TRAIN_FRACTION is whatever remains (~0.70).

CLASS_NAMES = ["NORMAL", "INSTABILITY", "FALL"]
WINDOW_SIZE = 100
WINDOW_STEP = 50
TARGET_RATE_HZ = 50

# Official published cohort sizes, for the Methods table's "Original
# subjects" column -- distinct from "subjects used", which is whatever this
# run actually found under ml/data/raw/. If these ever disagree, the
# manifest's `subject_count_mismatch` flag below is what should be believed.
SISFALL_OFFICIAL_SUBJECTS = 38  # Sucerquia et al. 2017: 23 young (SA) + 15 elderly (SE)
KFALL_OFFICIAL_SUBJECTS = 32    # Yu et al. 2021: SA06-SA38, missing SA34


# ===========================================================================
# PER-SOURCE SUBJECT COLLECTION
# ===========================================================================


def _sisfall_subjects() -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """One (windows, labels) pair per SisFall subject folder.

    Reuses `preprocess.load_sisfall_file` / `.downsample` / `.create_windows`
    / `.get_label_from_filename` exactly as `preprocess.process_all_files()`
    does per file; the only difference is grouping by subject instead of by
    a hardcoded TEST_SUBJECTS set.
    """
    if not sisfall_pp.DATASET_DIR.exists():
        raise FileNotFoundError(
            f"SisFall dataset not found at {sisfall_pp.DATASET_DIR}"
        )

    per_subject: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for subject_dir in sorted(sisfall_pp.DATASET_DIR.iterdir()):
        if not subject_dir.is_dir():
            continue
        subject_id = subject_dir.name
        X_list, y_list = [], []
        for filepath in sorted(subject_dir.glob("*.txt")):
            if filepath.name.lower() in ("readme.txt", "desktop.ini"):
                continue
            data = sisfall_pp.load_sisfall_file(filepath)
            if data is None or len(data) < sisfall_pp.DOWNSAMPLE_FACTOR * WINDOW_SIZE:
                continue
            data = sisfall_pp.downsample(data, sisfall_pp.DOWNSAMPLE_FACTOR)
            windows = sisfall_pp.create_windows(data, WINDOW_SIZE, WINDOW_STEP)
            if len(windows) == 0:
                continue
            label = sisfall_pp.get_label_from_filename(filepath.name)
            if label == -1:
                continue
            X_list.append(windows)
            y_list.append(np.full(len(windows), label, dtype=np.int32))
        if X_list:
            per_subject[subject_id] = (
                np.concatenate(X_list, axis=0).astype(np.float32),
                np.concatenate(y_list, axis=0),
            )
    return per_subject


def _kfall_subjects() -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """One (windows, labels) pair per KFall subject folder.

    Reuses `preprocess_kfall.process_subject()` unchanged -- it already
    returns exactly this shape for one subject.
    """
    if not kfall_pp.KFALL_DIR.exists():
        raise FileNotFoundError(f"KFall dataset not found at {kfall_pp.KFALL_DIR}")

    per_subject: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for subject_dir in sorted(kfall_pp.KFALL_DIR.iterdir()):
        if not subject_dir.is_dir():
            continue
        X_list, y_list = kfall_pp.process_subject(subject_dir)
        if X_list:
            per_subject[subject_dir.name] = (
                np.array(X_list, dtype=np.float32),
                np.array(y_list, dtype=np.int32),
            )
    return per_subject


def _vestibular_subjects() -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """One (windows, labels) pair per vestibular-study participant folder.

    Reuses `preprocess_vestibular.process_participant()` unchanged.
    """
    if not vest_pp.VEST_DIR.exists():
        raise FileNotFoundError(f"Vestibular dataset not found at {vest_pp.VEST_DIR}")

    per_subject: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
    for p_dir in sorted(vest_pp.VEST_DIR.iterdir()):
        if not p_dir.is_dir():
            continue
        windows, labels, _file_count, _skipped = vest_pp.process_participant(p_dir)
        if windows is not None and len(windows) > 0:
            per_subject[p_dir.name] = (windows.astype(np.float32), labels)
    return per_subject


# ===========================================================================
# SUBJECT-INDEPENDENT SPLIT
# ===========================================================================


def split_subject_ids(
    subject_ids: List[str], *, seed: int, val_fraction: float, test_fraction: float
) -> Tuple[List[str], List[str], List[str]]:
    """Deterministically assign whole subjects to train/val/test.

    Every subject's windows go entirely to one split -- this is the one
    property the prior pipeline violated for none of its splits (SisFall and
    the vestibular dataset were already subject-disjoint train/test; the
    defect was the *absence* of a val split and KFall's all-into-train
    routing, not within-subject leakage). Rounds val/test up to at least one
    subject each so a small source is never silently absorbed into train.
    """
    ids = sorted(subject_ids)
    n = len(ids)
    if n < 3:
        raise ValueError(
            f"cannot make a 3-way subject split from only {n} subject(s): {ids}"
        )
    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    shuffled = [ids[i] for i in order]

    n_val = max(1, round(n * val_fraction))
    n_test = max(1, round(n * test_fraction))
    n_train = n - n_val - n_test
    if n_train < 1:
        raise ValueError(
            f"val_fraction+test_fraction leave no train subjects for n={n}"
        )

    train = shuffled[:n_train]
    val = shuffled[n_train : n_train + n_val]
    test = shuffled[n_train + n_val :]
    assert sorted(train + val + test) == ids
    return train, val, test


# ===========================================================================
# ASSEMBLY
# ===========================================================================


def _stack(per_subject: Dict[str, Tuple[np.ndarray, np.ndarray]], subject_ids: List[str]):
    if not subject_ids:
        return np.empty((0, WINDOW_SIZE, 7), dtype=np.float32), np.empty((0,), dtype=np.int32)
    X = np.concatenate([per_subject[s][0] for s in subject_ids], axis=0)
    y = np.concatenate([per_subject[s][1] for s in subject_ids], axis=0)
    return X, y


def _class_counts(y: np.ndarray) -> Dict[str, int]:
    return {name: int((y == i).sum()) for i, name in enumerate(CLASS_NAMES)}


def build() -> None:
    print("=" * 70)
    print("  BUILD SUBJECT-INDEPENDENT 3-CLASS DATASET (v2)")
    print("=" * 70)

    print("\n[1/3] Loading per-subject windows...")
    sources = {
        "sisfall": _sisfall_subjects(),
        "kfall": _kfall_subjects(),
        "vestibular": _vestibular_subjects(),
    }
    for name, per_subject in sources.items():
        total = sum(len(y) for _X, y in per_subject.values())
        print(f"  {name:12s}: {len(per_subject):3d} subjects, {total:7,d} windows")

    print("\n[2/3] Splitting subjects (train/val/test), source-stratified...")
    splits: Dict[str, Dict[str, List[str]]] = {}
    for name, per_subject in sources.items():
        train_ids, val_ids, test_ids = split_subject_ids(
            list(per_subject.keys()),
            seed=SPLIT_SEED,
            val_fraction=VAL_FRACTION,
            test_fraction=TEST_FRACTION,
        )
        splits[name] = {"train": train_ids, "val": val_ids, "test": test_ids}
        print(
            f"  {name:12s}: train={len(train_ids):3d}  val={len(val_ids):3d}  "
            f"test={len(test_ids):3d}"
        )

    print("\n[3/3] Assembling and saving...")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_sources = {}
    final = {}
    for split_name in ("train", "val", "test"):
        X_parts, y_parts, origin_parts = [], [], []
        for source_name, per_subject in sources.items():
            ids = splits[source_name][split_name]
            X, y = _stack(per_subject, ids)
            X_parts.append(X)
            y_parts.append(y)
            origin_parts.append(np.full(len(y), source_name))
        X_all = np.concatenate(X_parts, axis=0)
        y_all = np.concatenate(y_parts, axis=0)
        origin_all = np.concatenate(origin_parts, axis=0)

        # Shuffle within the split so no source forms a contiguous block --
        # purely cosmetic for batch composition, never crosses split boundaries.
        rng = np.random.default_rng(SPLIT_SEED + {"train": 0, "val": 1, "test": 2}[split_name])
        perm = rng.permutation(len(X_all))
        X_all, y_all, origin_all = X_all[perm], y_all[perm], origin_all[perm]

        final[split_name] = (X_all, y_all)
        np.savez_compressed(OUTPUT_DIR / f"X_{split_name}.npz", data=X_all)
        np.savez_compressed(OUTPUT_DIR / f"y_{split_name}.npz", data=y_all)
        # Per-window source dataset ('sisfall'/'kfall'/'vestibular') -- not
        # consumed by ordinary training, only by the Finding-4 domain-origin
        # confound check (does a classifier trained on this label alone do
        # suspiciously well, which would suggest the 3-class model could be
        # keying on acquisition-pipeline artefacts rather than motion).
        np.savez_compressed(OUTPUT_DIR / f"origin_{split_name}.npz", data=origin_all)
        print(f"  {split_name}: {X_all.shape}  classes={_class_counts(y_all)}")

    for source_name, per_subject in sources.items():
        manifest_sources[source_name] = {
            "subjects_used": len(per_subject),
            "subjects_by_split": {
                split_name: sorted(ids) for split_name, ids in splits[source_name].items()
            },
            "windows_by_split": {
                split_name: _class_counts(_stack(per_subject, ids)[1])
                for split_name, ids in splits[source_name].items()
            },
        }

    manifest = {
        "schema_version": 2,
        "generated_by": "ml/training/build_subject_independent_dataset.py",
        "split_method": "subject-independent, source-stratified (each source split "
        f"independently at {1 - VAL_FRACTION - TEST_FRACTION:.0%}/"
        f"{VAL_FRACTION:.0%}/{TEST_FRACTION:.0%} train/val/test by subject count, "
        "then concatenated across sources)",
        "split_seed": SPLIT_SEED,
        "window_size_samples": WINDOW_SIZE,
        "window_step_samples": WINDOW_STEP,
        "target_rate_hz": TARGET_RATE_HZ,
        "class_names": CLASS_NAMES,
        "datasets": {
            "sisfall": {
                "original_subjects_published": SISFALL_OFFICIAL_SUBJECTS,
                "subjects_used": manifest_sources["sisfall"]["subjects_used"],
                "subject_count_mismatch": (
                    manifest_sources["sisfall"]["subjects_used"] != SISFALL_OFFICIAL_SUBJECTS
                ),
                "original_sampling_rate_hz": 200,
                "processing_applied": "decimation 200Hz->50Hz (every 4th sample), "
                "SVM channel appended, 2s/100-sample windows at 50% overlap",
                "classes_contributed": ["NORMAL", "FALL"],
                **manifest_sources["sisfall"],
            },
            "kfall": {
                "original_subjects_published": KFALL_OFFICIAL_SUBJECTS,
                "subjects_used": manifest_sources["kfall"]["subjects_used"],
                "subject_count_mismatch": (
                    manifest_sources["kfall"]["subjects_used"] != KFALL_OFFICIAL_SUBJECTS
                ),
                "original_sampling_rate_hz": 100,
                "processing_applied": "decimation 100Hz->50Hz (every 2nd sample), "
                "SVM channel appended, 2s/100-sample windows at 50% overlap; "
                "FALL windows selected from labelled onset/impact frames",
                "classes_contributed": ["NORMAL", "FALL"],
                "note": "KFall never contributes an INSTABILITY window in this "
                "pipeline -- its task routing is exactly ADL(<=T19)->NORMAL, "
                "fall tasks(>=T20)->FALL.",
                **manifest_sources["kfall"],
            },
            "vestibular": {
                "original_subjects_published": None,
                "subjects_used": manifest_sources["vestibular"]["subjects_used"],
                "subject_count_mismatch": False,
                "original_sampling_rate_hz": 128,
                "processing_applied": "anti-aliased polyphase resample 128Hz->50Hz "
                "(up=25/down=64, ml.common.resampling), SVM channel "
                "appended after resampling, 2s/100-sample windows at 50% overlap",
                "classes_contributed": ["NORMAL", "INSTABILITY"],
                "note": "Sole source of INSTABILITY windows in the merged corpus. "
                "This is a custom dataset collected by this study, not a public "
                "release -- 'original_subjects_published' does not apply.",
                **manifest_sources["vestibular"],
            },
        },
        "merged_totals": {
            split_name: _class_counts(final[split_name][1]) for split_name in ("train", "val", "test")
        },
    }

    manifest_path = OUTPUT_DIR / "dataset_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=False) + "\n", encoding="utf-8")
    print(f"\n  Manifest: {manifest_path}")
    print("\n  Methods-table summary:")
    print(f"  {'Dataset':<12} {'Orig. subj.':>11} {'Used':>5} {'Orig. Hz':>9} {'Classes':<20}")
    for name, info in manifest["datasets"].items():
        orig = info["original_subjects_published"]
        orig_str = str(orig) if orig is not None else "N/A (custom)"
        print(
            f"  {name:<12} {orig_str:>11} {info['subjects_used']:>5} "
            f"{info['original_sampling_rate_hz']:>9} {','.join(info['classes_contributed']):<20}"
        )

    print("\n" + "=" * 70)
    print("  DONE. Upload the whole processed_3class_v2/ directory to Drive.")
    print("=" * 70)


if __name__ == "__main__":
    try:
        build()
    except FileNotFoundError as exc:
        print(f"\nERROR: {exc}", file=sys.stderr)
        sys.exit(1)


In [ ]:
# Cell 9 -- Symlink 3 thu muc raw tren Drive vao dung vi tri /content/repo cho

def _relink(link_path: Path, target: str):
    if link_path.is_symlink() or link_path.exists():
        if link_path.is_dir() and not link_path.is_symlink():
            raise RuntimeError(f'{link_path} da la thu muc that (khong phai symlink) -- kiem tra lai.')
        link_path.unlink()
    os.symlink(target, link_path, target_is_directory=True)
    print(f'{link_path}  ->  {target}')

_relink(REPO / 'ml' / 'data' / 'raw' / 'SisFall_dataset',    DRIVE_SISFALL_DIR)
_relink(REPO / 'ml' / 'data' / 'raw' / 'KFALL_dataset',      DRIVE_KFALL_DIR)
_relink(REPO / 'ml' / 'data' / 'raw' / 'vestibular_dataset', DRIVE_VESTIBULAR_DIR)


In [ ]:
# Cell 10 -- Chay build() that su
import sys
sys.path.insert(0, str(REPO))

from ml.training.build_subject_independent_dataset import build

build()


In [ ]:
# Cell 11 -- Chep ket qua processed_3class_v2/ nguoc len Drive
import shutil

LOCAL_OUTPUT_DIR = REPO / 'ml' / 'data' / 'processed_3class_v2'
drive_out = Path(DRIVE_OUTPUT_DIR)
drive_out.mkdir(parents=True, exist_ok=True)

for f in sorted(LOCAL_OUTPUT_DIR.glob('*')):
    dest = drive_out / f.name
    shutil.copy2(f, dest)
    print(f'  {f.name}  ({dest.stat().st_size/1024/1024:.2f} MB)  -> {dest}')

print(f'\nDa chep xong vao {DRIVE_OUTPUT_DIR}')
print('Buoc tiep theo: mo train_models_colab_v2.ipynb, chay tu Cell 1 -- '
      'Cell 2 cua notebook do se tu kiem tra day du file tai chinh duong dan nay.')
